In [1]:
import pandas as pd
import numpy as np

In [5]:
import yfinance as yf

# 25 diverse stocks across major economic sectors
tickers = [
    "AAPL", "MSFT", "GOOGL", "NVDA", "META",  # Tech / Communication
    "JPM", "BAC", "GS",                      # Finance
    "JNJ", "PFE", "UNH",                     # Healthcare
    "XOM", "CVX",                            # Energy
    "WMT", "AMZN", "COST",                   # Retail / Consumer Discretionary
    "PG", "KO", "PEP",                       # Consumer Staples
    "CAT", "GE",                             # Industrials
    "NEE",                                   # Utilities
    "AMT",                                   # Real Estate
    "LIN",                                   # Materials
]


# Download only the Closing prices to keep the DataFrame small and clean
data = yf.download(tickers, start="2023-01-01", end="2026-01-01")["Close"]



[*********************100%***********************]  24 of 24 completed


In [11]:

portfolio_df = data

portfolio_df.head()


Ticker,AAPL,AMT,AMZN,BAC,CAT,COST,CVX,GE,GOOGL,GS,...,META,MSFT,NEE,NVDA,PEP,PFE,PG,UNH,WMT,XOM
Date,,,,,,,,,,,,,,,,,,,,,
2023-01-03,122.876747,190.250671,85.820000,30.485411,225.586609,433.134644,149.220367,52.000916,88.279961,316.461731,...,123.556602,232.510544,75.184204,14.267299,157.697617,40.977406,137.454956,481.460876,45.855385,94.180984
2023-01-04,124.144104,194.088135,85.139999,31.058540,227.928619,436.268890,147.633713,55.027588,87.249771,317.814575,...,126.161659,222.339798,75.776131,14.699852,157.310883,40.074081,138.053467,468.334473,45.906456,94.455093
2023-01-05,122.827614,187.999588,83.120003,30.994864,226.908722,430.182007,150.292389,55.882000,85.387474,314.213196,...,125.735741,215.750107,74.107956,14.217464,155.667191,39.698360,136.339462,454.836792,45.750004,96.568443
2023-01-06,127.346924,193.645004,86.080002,31.304182,235.011246,461.409515,151.424484,56.391521,86.516724,318.161896,...,128.786514,218.292816,75.022766,14.809487,159.183121,40.705605,139.586121,454.873962,46.870838,97.735641
2023-01-09,127.867661,194.353989,87.360001,30.831112,232.895859,457.472656,150.240952,56.963753,87.190323,322.658966,...,128.241730,220.418213,75.399467,15.575925,157.627319,38.683117,137.881180,454.929657,46.286465,95.914101


In [66]:
from pypfopt import expected_returns, risk_models
from pypfopt.efficient_frontier import EfficientFrontier

# 1. Calculate returns and Ledoit-Wolf covariance matrix
mu = expected_returns.mean_historical_return(portfolio_df)
S = risk_models.risk_matrix(portfolio_df, method="ledoit_wolf")

# 2. FIX: Change bounds to allow negative weights (Short Selling)
# Option A: Unconstrained shorting -> bounds=(None, None)
# Option B: Capped shorting (e.g., max 30% short per asset) -> bounds=(-0.30, 1.0)
# Define your current Risk-Free Rate
current_rf = 0.045  
ef = EfficientFrontier(mu, S, (-0.25, 1), solver="SCS")   # (-0.25,1) = short selling allowed , (0,1) = no short selling
 

# 3. Optimize for Maximum Sharpe Ratio
weights = ef.max_sharpe(risk_free_rate = current_rf)
cleaned_weights = ef.clean_weights()

# Print both Long and Short positions
print("\nOptimal Portfolio Weights:")
print("-" * 25)
for ticker, weight in cleaned_weights.items():
    if weight != 0:  # Show both positive (long) and negative (short) allocations
        position_type = "LONG" if weight > 0 else "SHORT"
        print(f"{ticker}: {weight:.2%} ({position_type})")
print()

print("\nPortfolio Performance")
print("-" * 25)
ef.portfolio_performance(verbose=True)
print()



Optimal Portfolio Weights:
-------------------------
AAPL: -6.20% (SHORT)
AMT: 2.90% (LONG)
AMZN: -17.81% (SHORT)
BAC: -20.92% (SHORT)
CAT: 0.21% (LONG)
COST: -18.02% (SHORT)
CVX: -25.00% (SHORT)
GE: 45.66% (LONG)
GOOGL: 19.53% (LONG)
GS: 2.73% (LONG)
JNJ: 49.20% (LONG)
JPM: 19.97% (LONG)
KO: 32.92% (LONG)
LIN: -25.00% (SHORT)
META: 18.62% (LONG)
MSFT: -25.00% (SHORT)
NEE: -10.62% (SHORT)
NVDA: 33.29% (LONG)
PEP: -8.07% (SHORT)
PFE: -25.00% (SHORT)
PG: -5.92% (SHORT)
UNH: -7.29% (SHORT)
WMT: 49.54% (LONG)
XOM: 20.26% (LONG)


Portfolio Performance
-------------------------
Expected annual return: 118.1%
Annual volatility: 26.7%
Sharpe Ratio: 4.25



C:\Users\iamby\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\pypfopt\efficient_frontier\efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
  warnings.warn(


"""


By default, PyPortfolioOpt assumes your portfolio_df contains daily prices. 
Therefore, it automatically uses a parameter of frequency=252 (the standard number of trading days in a year) 
to annualize both the returns and the covariance matrix.

Here is what is happening under the hood:

mu: It calculates the daily return and annualizes it using frequency=252.

S: It calculates the daily covariance matrix and multiplies it by frequency=252.







# If portfolio_df contains MONTHLY data:

mu = expected_returns.mean_historical_return(portfolio_df, frequency=12)
<br>
S = risk_models.risk_matrix(portfolio_df, method="ledoit_wolf", frequency=12)






Summary of frequency settings:

Daily data: frequency=252 (Default, no need to type it)

Monthly data: frequency=12 (You must specify this)

Weekly data: frequency=52 (You must specify this)








"""

In [48]:
print("Variance - Covariance Matrix")
print("-" * 50)
print("\n")
S.head()

Variance - Covariance Matrix
--------------------------------------------------




Ticker,AAPL,AMT,AMZN,BAC,CAT,COST,CVX,GE,GOOGL,GS,...,META,MSFT,NEE,NVDA,PEP,PFE,PG,UNH,WMT,XOM
Ticker,,,,,,,,,,,,,,,,,,,,,
AAPL,0.065698,0.005949,0.036386,0.020013,0.023454,0.017355,0.013499,0.021622,0.034043,0.024858,...,0.038900,0.027754,0.009261,0.045808,0.006584,0.010199,0.006010,0.002206,0.013013,0.010228
AMT,0.005949,0.064485,-0.001167,0.008967,0.003934,0.006427,0.003422,0.000817,-0.002282,0.005773,...,-0.001773,0.001845,0.026662,-0.013328,0.013797,0.016647,0.012571,0.003349,0.006911,0.003805
AMZN,0.036386,-0.001167,0.100470,0.023852,0.028699,0.020421,0.009528,0.032249,0.051755,0.032566,...,0.070888,0.042958,0.005468,0.071076,-0.002174,0.006342,-0.000001,0.001349,0.013586,0.004271
BAC,0.020013,0.008967,0.023852,0.065135,0.035591,0.011211,0.022991,0.025613,0.017211,0.048188,...,0.023972,0.012572,0.013257,0.024181,0.003443,0.012512,0.003970,0.007211,0.011567,0.020605
CAT,0.023454,0.003934,0.028699,0.035591,0.083210,0.008775,0.024488,0.032609,0.022763,0.043509,...,0.029862,0.018415,0.008561,0.043410,0.002082,0.012215,-0.000517,0.002947,0.007199,0.022499


In [60]:
# Try optimizing for lowest risk instead of highest Sharpe ratio
ef1 = EfficientFrontier(mu, S,(-0.25,1), solver="SCS")   #(0.25,1) = short selling allowed ,(0,1) = no short selling
weights1 = ef1.min_volatility()
cleaned_weights1 = ef1.clean_weights()


In [59]:
# Print both Long and Short positions
print("\nOptimal Portfolio Weights:")
print("-" * 25)
for ticker, weight in cleaned_weights1.items():
    if weight != 0:  # Show both positive (long) and negative (short) allocations
        position_type = "LONG" if weight > 0 else "SHORT"
        print(f"{ticker}: {weight:.2%} ({position_type})")
print()

print("\nPortfolio Performance")
print("-" * 25)
ef1.portfolio_performance(verbose=True)
print()



Optimal Portfolio Weights:
-------------------------
AAPL: -3.45% (SHORT)
AMT: 3.73% (LONG)
AMZN: 0.45% (LONG)
BAC: -0.30% (SHORT)
CAT: 2.06% (LONG)
COST: 4.06% (LONG)
CVX: 2.53% (LONG)
GE: 2.39% (LONG)
GOOGL: 5.24% (LONG)
GS: -3.96% (SHORT)
JNJ: 16.00% (LONG)
JPM: 3.65% (LONG)
KO: 18.72% (LONG)
LIN: 0.59% (LONG)
META: -0.77% (SHORT)
MSFT: 10.12% (LONG)
NEE: -1.69% (SHORT)
NVDA: 3.02% (LONG)
PEP: 4.35% (LONG)
PFE: 1.28% (LONG)
PG: 10.96% (LONG)
UNH: 4.18% (LONG)
WMT: 6.64% (LONG)
XOM: 10.19% (LONG)


Portfolio Performance
-------------------------
Expected annual return: 16.5%
Annual volatility: 9.8%
Sharpe Ratio: 1.67



# Black-Litterman Model
#### The Black-Litterman model blends Market Equilibrium (what the market thinks returns will be based on asset market caps) with Investor Views (your personal predictions or assumptions) to create a stabilized, adjusted return vector (mu_bl).To run this, you need a proxy for market caps (like total outstanding shares \(\times \) price) or a benchmark index weight (like the S&P 500) to find the starting baseline.

In [74]:
import pandas as pd
from pypfopt import black_litterman, BlackLittermanModel
from pypfopt import expected_returns, risk_models
from pypfopt.efficient_frontier import EfficientFrontier

# 1. Base Setup (Use your working elements)
S = risk_models.risk_matrix(portfolio_df, method="ledoit_wolf")

# 2. Setup Market Cap Data (REQUIRED for market-implied prior returns)
# Replace these arbitrary proxy values with your portfolio assets' actual market caps
market_caps = {col: 1000000000 for col in portfolio_df.columns} 

# 3. Calculate Market Prior 
# Pass a benchmark price series (e.g. SPY/S&P500) into risk aversion. 
# Alternatively, use the default financial constant proxy of 2.5 directly.
delta = 2.5 

# FIX: Changed method name to market_implied_prior_returns
prior = black_litterman.market_implied_prior_returns(market_caps, delta, S)

# 4. Define your unique investor views
views = {
    portfolio_df.columns[0]: 0.12,  # Example: 12% return for your first stock
    portfolio_df.columns[1]: 0.08   # Example: 8% return for your second stock
}

# 5. Inject views into Black-Litterman
bl = BlackLittermanModel(S, pi=prior, absolute_views=views)
mu_bl = bl.bl_returns()  # Compiles adjusted expected return vector

# 6. Feed back into the Efficient Frontier
ef_bl = EfficientFrontier(mu_bl, S, (-0.25,1), solver="SCS")   # (-0.25,1) = short selling allowed
weights_bl = ef_bl.max_sharpe()
cleaned_weights_bl = ef_bl.clean_weights()



In [75]:
# Print both Long and Short positions
print("\nOptimal Portfolio Weights:")
print("-" * 25)
for ticker, weight in cleaned_weights_bl.items():
    if weight != 0:  # Show both positive (long) and negative (short) allocations
        position_type = "LONG" if weight > 0 else "SHORT"
        print(f"{ticker}: {weight:.2%} ({position_type})")
print()

print("\nPortfolio Performance")
print("-" * 25)
ef_bl.portfolio_performance(verbose=True)
print()


Optimal Portfolio Weights:
-------------------------
AAPL: 18.09% (LONG)
AMT: 15.47% (LONG)
AMZN: 3.02% (LONG)
BAC: 3.02% (LONG)
CAT: 3.02% (LONG)
COST: 3.02% (LONG)
CVX: 3.02% (LONG)
GE: 3.02% (LONG)
GOOGL: 3.02% (LONG)
GS: 3.02% (LONG)
JNJ: 3.02% (LONG)
JPM: 3.02% (LONG)
KO: 3.02% (LONG)
LIN: 3.02% (LONG)
META: 3.02% (LONG)
MSFT: 3.02% (LONG)
NEE: 3.02% (LONG)
NVDA: 3.02% (LONG)
PEP: 3.02% (LONG)
PFE: 3.02% (LONG)
PG: 3.02% (LONG)
UNH: 3.02% (LONG)
WMT: 3.02% (LONG)
XOM: 3.02% (LONG)


Portfolio Performance
-------------------------
Expected annual return: 5.7%
Annual volatility: 12.8%
Sharpe Ratio: 0.44



In [76]:
print("--- Black-Litterman Weights ---")
print(cleaned_weights_bl)
print()
ef_bl.portfolio_performance(verbose=True)
print()


--- Black-Litterman Weights ---
OrderedDict({'AAPL': 0.18094, 'AMT': 0.15469, 'AMZN': 0.0302, 'BAC': 0.0302, 'CAT': 0.0302, 'COST': 0.0302, 'CVX': 0.0302, 'GE': 0.0302, 'GOOGL': 0.0302, 'GS': 0.0302, 'JNJ': 0.0302, 'JPM': 0.0302, 'KO': 0.0302, 'LIN': 0.0302, 'META': 0.0302, 'MSFT': 0.0302, 'NEE': 0.0302, 'NVDA': 0.0302, 'PEP': 0.0302, 'PFE': 0.0302, 'PG': 0.0302, 'UNH': 0.0302, 'WMT': 0.0302, 'XOM': 0.0302})

Expected annual return: 5.7%
Annual volatility: 12.8%
Sharpe Ratio: 0.44



 # Hierarchical Risk Parity (HRP)
 #### HRP completely ignores the expected returns vector (mu) and uses machine learning (hierarchical clustering) on your covariance matrix. It does not require a solver like SCS, meaning it will never throw an optimization error.

In [78]:
# 1. FIX: Patch the missing SciPy variable for Python 3.14 / SciPy 1.18+
import scipy.cluster.hierarchy as sch
sch._LINKAGE_METHODS = ["single", "complete", "average", "weighted", "centroid", "median", "ward"]

# 2. Now import your PyPortfolioOpt modules safely
from pypfopt.hierarchical_portfolio import HRPOpt
from pypfopt import expected_returns

# 3. Rest of your working HRP code
returns = expected_returns.returns_from_prices(portfolio_df).dropna()

hrp = HRPOpt(returns)
weights_hrp = hrp.optimize()
cleaned_weights_hrp = hrp.clean_weights()

In [79]:
# Print both Long and Short positions
print("\nOptimal Portfolio Weights:")
print("-" * 25)
for ticker, weight in cleaned_weights_hrp.items():
    if weight != 0:  # Show both positive (long) and negative (short) allocations
        position_type = "LONG" if weight > 0 else "SHORT"
        print(f"{ticker}: {weight:.2%} ({position_type})")
print()

print("\nPortfolio Performance")
print("-" * 25)
hrp.portfolio_performance(verbose=True)
print()


Optimal Portfolio Weights:
-------------------------
AAPL: 3.30% (LONG)
AMT: 4.56% (LONG)
AMZN: 2.53% (LONG)
BAC: 2.93% (LONG)
CAT: 2.10% (LONG)
COST: 6.75% (LONG)
CVX: 4.77% (LONG)
GE: 2.17% (LONG)
GOOGL: 1.95% (LONG)
GS: 1.48% (LONG)
JNJ: 6.07% (LONG)
JPM: 2.06% (LONG)
KO: 9.68% (LONG)
LIN: 6.72% (LONG)
META: 1.62% (LONG)
MSFT: 3.31% (LONG)
NEE: 3.61% (LONG)
NVDA: 1.04% (LONG)
PEP: 7.54% (LONG)
PFE: 3.07% (LONG)
PG: 7.81% (LONG)
UNH: 2.87% (LONG)
WMT: 7.17% (LONG)
XOM: 4.88% (LONG)


Portfolio Performance
-------------------------
Expected annual return: 16.1%
Annual volatility: 10.8%
Sharpe Ratio: 1.49



In [80]:
print("--- HRP Weights ---")
print(cleaned_weights_hrp)
print()
hrp.portfolio_performance(verbose=True)
print()

--- HRP Weights ---
OrderedDict({'AAPL': 0.03301, 'AMT': 0.04558, 'AMZN': 0.02532, 'BAC': 0.02926, 'CAT': 0.02102, 'COST': 0.06753, 'CVX': 0.04771, 'GE': 0.02169, 'GOOGL': 0.01947, 'GS': 0.01484, 'JNJ': 0.06069, 'JPM': 0.02057, 'KO': 0.09675, 'LIN': 0.06719, 'META': 0.0162, 'MSFT': 0.03313, 'NEE': 0.03613, 'NVDA': 0.01045, 'PEP': 0.07538, 'PFE': 0.03066, 'PG': 0.07815, 'UNH': 0.02873, 'WMT': 0.07173, 'XOM': 0.0488})

Expected annual return: 16.1%
Annual volatility: 10.8%
Sharpe Ratio: 1.49

